# Model comparison and calibration

**Recipe 50** · Level 4 (Advanced) · Decision type: `Choice` + `Noul` + `Score`

Compare Jev and alternative decision backends on identical labeled tasks, calibrating each
model separately and measuring held-out quality, latency, and cost.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S08: [TypeSafe AI: System One Adapter](https://github.com/typesafe-ai/system-one-adapter-python)

## What you will build

A harness that sends the same vendor compliance statement through the same typed interface
([the `Backend` protocol](../../docs/backends.md)) to three decision backends: Jev, and two
fabricated decision adapters, `atlas-compact` and `northstar-7b`. Each statement gets one
always-sent request (a `Choice` category and two `Noul` propositions, together) and, only when
the category answer calls for it, a dependent second request (a `Score` rating how specific the
statement is). Python combines the typed answers into one outcome per backend, and each backend
is calibrated separately, on its own validation answers.

**No live run has been recorded for this recipe.** Every number below comes from the offline
replay of hand-written, synthetic answers. That makes this notebook a harness check: it shows
the plumbing works -- the questions get built, dispatched identically to every backend, and
turned into typed answers and a composed outcome, without crashing. **It is not a comparison.**
No cell below ranks the three backends, states a winner, or reports a margin between them, and
none should be read as one: see "What was and was not measured" for exactly what that means, and
this recipe's README for how a real comparison would be produced.

## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. Three
backends answer identically through `decide(state, questions)`
([the `Backend` protocol](../../docs/backends.md)), but only `jev`'s construction is sensitive to
`JEV_COOKBOOK_LIVE`: the other two are fabricated decision adapters this recipe never calls live,
so they are always built as plain replay backends, whatever the environment says. The header
below describes `jev`'s own mode; the line beneath it names every backend and its own mode, so
switching `jev` to live never silently changes what `atlas-compact` and `northstar-7b` are.

In [ ]:
from jev_cookbook import ReplayBackend, get_backend, load_helpers
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_confusion_matrix,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()

# The harness dispatches to every backend through the same Backend protocol. "jev" is the only
# one this recipe ever calls live; atlas-compact and northstar-7b are fabricated decision
# adapters built as plain replay backends regardless of JEV_COOKBOOK_LIVE.
BACKEND_TAGS = {"jev": None, "atlas-compact": "atlas", "northstar-7b": "north"}
backends = {
    name: (
        get_backend(fixtures=responses_path())
        if name == "jev"
        else ReplayBackend.from_json(responses_path(tag=tag))
    )
    for name, tag in BACKEND_TAGS.items()
}

# scored excludes the three demo examples: they carry no gold label, so no metric below is
# about them (see "gold label" under Evaluation).
scored = [e for e in examples if e.split not in ("train", "demo")]

offline = backends["jev"].mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
selection = " (a selection step, not a reported result)"

run_header(
    50,
    "Model comparison and calibration",
    backend=backends["jev"],
    n_examples=len(scored),
)
print("Every backend in this harness, and its own run mode:")
for name, backend in backends.items():
    print(f"  {name:14s} mode={backend.mode}")

## The state

The state Jev sees is the statement's text alone, built by `helpers.build_state`. Each input's
`fields` also carries `resubmission`, a flag Python keeps for itself (the stateful rule in
"Python's part" below) and never sends to any backend.

In [ ]:
demo = {e.id: e for e in examples if e.split == "demo"}
d01 = demo["d01"]
print("Python has:")
print(d01.fields)
state = helpers.build_state(d01.fields)
print("Every backend sees:")
print(state)

## The questions

Two requests make up one task. Request A is always sent: three independent judgments about one
statement, asked together -- a `Choice` over the four [compliance categories](../../docs/glossary.md#choice)
(with `other` as the explicit fallback, CONTRIBUTING.md section 3), and two
[`Noul`](../../docs/glossary.md#noul) propositions, each a statement that can be true or false,
never a question. Request B asks a [`Score`](../../docs/glossary.md#score) rubric with every
level described in words, in order; it is sent only when request A's own `category` answer says
the statement is not `other` -- a question that depends on an earlier answer, in a later
request, never guessed at alongside the independent ones (CONTRIBUTING.md section 3). "Python's
part" below shows exactly where that boundary is enforced.

In [ ]:
questions_a = helpers.build_questions_a()
print("Request A (always sent, together):")
for name, question in questions_a.items():
    print(f"  {name} ({question.type}): {question.to_dict()}")

questions_b = helpers.build_questions_b()
print("\nRequest B (sent only when needs_followup(category_answer) is True):")
for name, question in questions_b.items():
    print(f"  {name} ({question.type}):")
    for level, text in enumerate(question.criteria):
        print(f"    {level}: {text}")

## One answer up close

Before any aggregate, look at one typed answer from each primitive, for one statement, from
`jev`. The `Choice` answer holds the chosen category, a probability for every option, and a
confidence rescaled by the number of options. A `Noul` answer is a probability of yes with no
native confidence field -- `jev_cookbook.evaluation.noul_confidence` derives one,
`|2p - 1|` (docs/glossary.md#confidence) -- used below to calibrate the two-sided gate.

In [ ]:
result_a = backends["jev"].decide(state, questions_a)
for name in ("category", "verifiable", "names_control"):
    show_answer(result_a[name])

In [ ]:
plot_answer_probabilities(result_a["category"], title="One Choice answer, from jev")

## The composition boundary

Request B depends on request A's own `category` answer: `helpers.needs_followup` is `False`
exactly when that answer is `other`. The boundary is visible here, not just asserted: the cell
below fetches request B for `d01` (whose `jev` category answer is not `other`) and explicitly
skips it for `d02` (whose answer is `other`) -- it never fetches and discards. Each backend
decides this from its **own** category answer, so two backends can disagree about whether a
given statement is worth a second request at all; the loop below shows what each backend, not
just `jev`, actually does for both demo statements.

In [ ]:
d02 = demo["d02"]
for example in (d01, d02):
    print(f"{example.id}: {example.fields['statement'][:60]!r}...")
    example_state = helpers.build_state(example.fields)
    for name, backend in backends.items():
        category_answer = backend.decide(example_state, questions_a)["category"]
        if helpers.needs_followup(category_answer):
            specificity_answer = backend.decide(example_state, questions_b)["specificity"]
            print(
                f"  {name:14s} category={category_answer.choice:20s} "
                f"-> fetches request B, specificity score {specificity_answer.score:.2f}"
            )
        else:
            print(
                f"  {name:14s} category={category_answer.choice:20s} "
                "-> skips request B (not worth asking)"
            )

## Python's part

`helpers.triage` combines one backend's answers into one outcome, and holds whatever the model
answers: a low-confidence `category` answer goes to [`review`](../../docs/glossary.md#review);
`other` is a complete answer (CONTRIBUTING.md section 4) that clears the statement once it has
passed that same confidence gate; otherwise both `Noul` propositions pass through a two-sided
[confidence gate](../../docs/glossary.md#confidence-gate) before Python's own arithmetic -- a
fixed business cutoff, `SPECIFICITY_CUTOFF`, never calibrated from data -- decides between
`flagged` and `needs_detail`. `d03` is a resubmission (state Python keeps outside the model):
a resubmission that still lands on `review` escalates instead, because it has already failed
this gate once before.

## Calibrating each backend separately

Two thresholds are chosen per backend, on `validation` only, then frozen:

- **The `Choice` confidence threshold**, `choice_threshold`, below which a `category` answer
  goes to review whatever it names. Chosen with `select_confidence_threshold`.
- **The `Noul` gate width**, `gate_width`, for the two-sided band
  `confidence = |2p - 1| >= gate_width`. A `Noul` has no native confidence
  (docs/glossary.md#confidence), so `noul_confidence` derives one from each stored probability,
  and the gate is chosen the same way a `Choice` confidence threshold is: on validation, by
  `select_confidence_threshold`, pooling both propositions' validation answers for one backend
  into one calibration (`verifiable` and `names_control` share one gate per backend, chosen
  together).

Both use `min_coverage` as the objective (not `target_accuracy=1.0`, which would force
validation accuracy to 1.0 by construction): the lowest-risk threshold among those that still
answer at least `CHOICE_MIN_COVERAGE` (respectively `GATE_MIN_COVERAGE`) of validation. Every
value printed below is a selection step, not a reported result, and every number in this
notebook is still a pipeline check: both labels are printed on every validation line.

In [ ]:
from jev_cookbook.evaluation import noul_confidence, select_confidence_threshold

CHOICE_MIN_COVERAGE = 0.80
GATE_MIN_COVERAGE = 0.80

validation = [e for e in examples if e.split == "validation"]

choice_threshold = {}
gate_width = {}
for name, backend in backends.items():
    cat_answers = [
        backend.decide(helpers.build_state(e.fields), questions_a)["category"] for e in validation
    ]
    cat_gold = [labels[e.id]["category"] for e in validation]
    correct = [a.choice == g for a, g in zip(cat_answers, cat_gold, strict=True)]
    confidence = [a.confidence for a in cat_answers]
    choice_threshold[name] = select_confidence_threshold(
        correct, confidence, min_coverage=CHOICE_MIN_COVERAGE
    )

    pooled_correct, pooled_conf = [], []
    for prop in ("verifiable", "names_control"):
        prop_answers = [
            backend.decide(helpers.build_state(e.fields), questions_a)[prop] for e in validation
        ]
        prop_gold = [labels[e.id][prop] for e in validation]
        probs = [a.noul for a in prop_answers]
        pooled_correct += [(p >= 0.5) == g for p, g in zip(probs, prop_gold, strict=True)]
        pooled_conf += noul_confidence(probs)
    gate_width[name] = select_confidence_threshold(
        pooled_correct, pooled_conf, min_coverage=GATE_MIN_COVERAGE
    )

    print(
        f"{name:14s} choice_threshold={choice_threshold[name]:.4f}{selection}{check}  "
        f"gate_width={gate_width[name]:.4f}{selection}{check}"
    )

In [ ]:
d03 = demo["d03"]
print(f"d03 ({d03.fields['resubmission']=}): {d03.fields['statement']}")
for name, backend in backends.items():
    example_state = helpers.build_state(d03.fields)
    category_answer = backend.decide(example_state, questions_a)["category"]
    verifiable_answer = backend.decide(example_state, questions_a)["verifiable"]
    names_control_answer = backend.decide(example_state, questions_a)["names_control"]
    specificity_answer = None
    if helpers.needs_followup(category_answer):
        specificity_answer = backend.decide(example_state, questions_b)["specificity"]
    routing = helpers.triage(
        d03.id,
        category_answer,
        verifiable_answer,
        names_control_answer,
        specificity_answer,
        choice_threshold=choice_threshold[name],
        gate_width=gate_width[name],
        resubmission=d03.fields["resubmission"],
    )
    print(
        f"  {name:14s} category={category_answer.choice:14s} "
        f"confidence={category_answer.confidence:.2f} -> {routing.outcome} ({routing.reason})"
    )

## Evaluation

Every number below is computed the same way for every backend, through the same functions, and
none is compared against another: this section reports what each backend's own frozen
calibration does, backend by backend, never a ranking. `jev_cookbook.evaluation.accuracy` scores
the `category` choice; `evaluate_selective` applies the frozen `choice_threshold`;
`evaluate_outcomes` reports the composed `triage` outcome, since its review branch is more than
a confidence gate (CONTRIBUTING.md section 4: the explicit `other` fallback and the two-sided
Noul gate are both unconditional branches of their own, not expressible as one confidence cut).
`test` is read once, after every threshold above is frozen.

In [ ]:
from jev_cookbook.evaluation import accuracy, evaluate_outcomes, evaluate_selective

test = [e for e in examples if e.split == "test"]


def decide_row(backend, example):
    state = helpers.build_state(example.fields)
    answers = backend.decide(state, questions_a)
    specificity_answer = None
    if helpers.needs_followup(answers["category"]):
        specificity_answer = backend.decide(state, questions_b)["specificity"]
    return answers, specificity_answer


for split_name, split_examples, label in (
    ("validation", validation, selection),
    ("test", test, ""),
):
    print(f"--- {split_name}, {len(split_examples)} examples{label}{check} ---")
    for name, backend in backends.items():
        rows = [decide_row(backend, e) for e in split_examples]
        cat_answers = [r[0]["category"] for r in rows]
        cat_gold = [labels[e.id]["category"] for e in split_examples]
        cat_acc = accuracy(cat_gold, cat_answers)
        sel = evaluate_selective(
            [a.choice == g for a, g in zip(cat_answers, cat_gold, strict=True)],
            [a.confidence for a in cat_answers],
            choice_threshold[name],
        )
        accepted, correct_composed, outcomes = [], [], []
        for (answers, specificity_answer), e in zip(rows, split_examples, strict=True):
            routing = helpers.triage(
                e.id,
                answers["category"],
                answers["verifiable"],
                answers["names_control"],
                specificity_answer,
                choice_threshold=choice_threshold[name],
                gate_width=gate_width[name],
                resubmission=e.fields["resubmission"],
            )
            outcomes.append(routing.outcome)
            is_review = routing.outcome in (helpers.REVIEW, helpers.ESCALATE_REVIEW)
            accepted.append(not is_review)
            correct_composed.append(answers["category"].choice == labels[e.id]["category"])
        composed = evaluate_outcomes(accepted, correct_composed)
        print(
            f"  {name:14s} category accuracy={cat_acc:.4f}{label}{check}  "
            f"category coverage={sel.coverage:.4f}{label}{check} risk={sel.risk:.4f}{label}{check}"
        )
        print(
            f"  {'':14s} composed coverage={composed.coverage:.4f}{label}{check} "
            f"composed risk={composed.risk:.4f}{label}{check}"
        )
        counts = {o: outcomes.count(o) for o in sorted(set(outcomes))}
        print(f"  {'':14s} outcomes: {counts}{check}")

### One backend's confusion matrix

A confusion matrix is a harness check, not a comparison: this one is `jev`'s own `category`
predictions on `test`, read on its own, to see which categories it confuses -- never set beside
the other two backends' matrices to imply a ranking.

In [ ]:
from jev_cookbook.evaluation import confusion_matrix

jev_test_cat = [decide_row(backends["jev"], e)[0]["category"] for e in test]
jev_test_gold = [labels[e.id]["category"] for e in test]
matrix = confusion_matrix(jev_test_gold, jev_test_cat, labels=list(helpers.CATEGORIES))
width = max(len(label) for label in matrix.labels)
leader_width = max(width + len("gold "), len("predicted ->"))
header = "  ".join(f"{label:>{width}s}" for label in matrix.labels)
print(f"jev, test confusion matrix, gold rows by predicted columns{check}:")
print(f"  {'predicted ->':<{leader_width}s}  {header}{check}")
for gold_label, row in zip(matrix.labels, matrix.matrix.tolist(), strict=True):
    counts = "  ".join(f"{c:{width}d}" for c in row)
    leader = f"gold {gold_label:<{width}s}"
    print(f"  {leader:<{leader_width}s}  {counts}{check}")
plot_confusion_matrix(matrix, title=f"jev, test split, {len(test)} examples{check}")

## Where the Noul gate bends

`confidence = |2p - 1|` is symmetric about 0.5, so the gate accepts on either side: a statement
is confidently true when `p >= (1 + g) / 2`, confidently false when `p <= (1 - g) / 2`, and sent
to review otherwise. Whether a given gate width `g` actually changes anything is not derivable
from `g` and a business threshold alone -- it depends on how the rule wires the gate into its
own composition, so it is measured here directly, against `helpers.triage`'s own composed
outcome, over a 101-point grid on `validation` (never `test`, which is read only once, above,
with the frozen width already applied).

In [ ]:
def composed_outcomes(backend, gate_width_value):
    outcomes = []
    for e in validation:
        answers, specificity_answer = decide_row(backend, e)
        routing = helpers.triage(
            e.id,
            answers["category"],
            answers["verifiable"],
            answers["names_control"],
            specificity_answer,
            choice_threshold=0.0,  # isolate the Noul gate: never review on category confidence here
            gate_width=gate_width_value,
            resubmission=e.fields["resubmission"],
        )
        outcomes.append(routing.outcome)
    return outcomes


grid = [i / 100 for i in range(101)]
print(f"gate width sweep on validation{selection}{check}, {len(grid)} points:")
for name, backend in backends.items():
    baseline = composed_outcomes(backend, 0.0)
    rows_cached = [decide_row(backend, e) for e in validation]
    first_bend = None
    changed_at_1 = 0
    for g in grid:
        current = composed_outcomes(backend, g)
        changed = sum(1 for a, b in zip(baseline, current, strict=True) if a != b)
        if changed and first_bend is None:
            first_bend = g
        if g == 1.0:
            changed_at_1 = changed
    print(
        f"  {name:14s} calibrated gate_width={gate_width[name]:.4f}{selection}{check}  "
        f"first bend at g={first_bend}{selection}{check}  "
        f"changed outcomes at g=1.0: {changed_at_1} of {len(validation)}{selection}{check}"
    )

## Leak check

The fixtures are synthetic and hand-written, so before reading any number above as telling us
anything about the harness's wiring, the cheapest surface features that could recover a gold
label without reading the statement at all are checked, at every granularity this recipe has a
label for: each `Noul` proposition, the `Score` cutoff, and the composed outcome. The signature
of a leak is a feature that reaches 1.0 with zero false positives on `validation` and transfers
unchanged to `test`; a probe against the composed outcome cannot detect a leak in one proposition
alone, because the other proposition's own errors keep the composed accuracy below 1.0 by
arithmetic, so each proposition is also checked on its own.

In [ ]:
import re

PROBES = {
    "has_digit": lambda s: bool(re.search(r"[0-9]", s)),
    "has_audit_word": lambda s: bool(re.search(r"audit|review(ed)?|ISO|SOC", s, re.IGNORECASE)),
}


def probe_report(name, feature, label_fn, split_examples):
    fires = fp = fn = tp = tn = 0
    for e in split_examples:
        gold = label_fn(labels[e.id])
        if gold is None:
            continue
        pred = feature(e.fields["statement"])
        fires += 1
        tp += pred and gold
        tn += (not pred) and (not gold)
        fp += pred and not gold
        fn += (not pred) and gold
    acc = (tp + tn) / fires if fires else float("nan")
    print(
        f"  {name:28s} fires={fires:2d} fp={fp} fn={fn} accuracy={acc:.4f}{check}"
        if fires
        else f"  {name:28s} fires=0 (measured nothing)"
    )


label_fns = {
    "verifiable": lambda gold: gold.get("verifiable"),
    "names_control": lambda gold: gold.get("names_control"),
    "specificity>=cutoff": lambda gold: (
        None
        if gold.get("specificity") is None
        else gold["specificity"] >= helpers.SPECIFICITY_CUTOFF
    ),
}

for split_name, split_examples in (("validation", validation), ("test", test)):
    print(f"-- {split_name}{check} --")
    for probe_name, feature in PROBES.items():
        for label_name, label_fn in label_fns.items():
            probe_report(f"{probe_name} -> {label_name}", feature, label_fn, split_examples)

print(f"-- composed (jev FLAGGED outcome), both splits{check} --")
for probe_name, feature in PROBES.items():
    fires = fp = fn = tp = tn = 0
    for e in validation + test:
        answers, specificity_answer = decide_row(backends["jev"], e)
        routing = helpers.triage(
            e.id,
            answers["category"],
            answers["verifiable"],
            answers["names_control"],
            specificity_answer,
            choice_threshold=choice_threshold["jev"],
            gate_width=gate_width["jev"],
            resubmission=e.fields["resubmission"],
        )
        gold_flagged = routing.outcome == helpers.FLAGGED  # jev's OWN composed outcome, not gold
        pred = feature(e.fields["statement"])
        fires += 1
        tp += pred and gold_flagged
        tn += (not pred) and (not gold_flagged)
        fp += pred and not gold_flagged
        fn += (not pred) and gold_flagged
    acc = (tp + tn) / fires if fires else float("nan")
    print(f"  {probe_name:28s} fires={fires:2d} fp={fp} fn={fn} accuracy={acc:.4f}{check}")

## What was and was not measured

No live run has been recorded for this recipe: `JEV_COOKBOOK_LIVE` was unset for every number
above, so every backend answered from stored, synthetic fixtures. The cell below states that
plainly and prints N -- the number of scored examples -- in both branches, as every recipe's
version of this cell does.

**The harness ran and produced tables; none of those cells is evidence about any model's
quality, latency, or cost.** Jev's own quality, latency, and cost are not stated anywhere above,
and neither are `atlas-compact`'s or `northstar-7b`'s: fabricated names describing a fabricated
decision adapter, used only to exercise the same `Backend` protocol offline. No accuracy number
above ranks the three backends; each is a pipeline check for the backend it names, nothing more.

In [ ]:
if offline:
    print(
        f"Provenance: {backends['jev'].mode}. Not measured live: a pipeline check, not a Jev result."
    )
    print(f"N: {len(validation)} validation and {len(test)} test examples{check}")
else:
    dates = ", ".join(getattr(backends["jev"], "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backends['jev'].mode}, model {backends['jev'].model}")
    print(f"Captured: {dates}")
    print(f"N: {len(validation)} validation and {len(test)} test examples")

## Next steps

- Change one Score level's wording in `helpers.py` and run `pytest recipes/50-model-comparison-and-calibration`
  or this notebook: the stored keys no longer match request B's question, so replay misses
  instead of guessing.
- Add a fourth backend by writing `fixtures/responses-<tag>.json` for the same replay keys and
  adding it to `BACKEND_TAGS` -- nothing else in the harness changes, because every backend
  answers through the same `Backend` protocol.
- Record a real live run (see "Switch to live" below) and watch every `check` label above
  disappear from the `jev` rows -- and notice that `atlas-compact` and `northstar-7b` stay
  labelled `synthetic` regardless, because this recipe never calls either of them live.
- Neighbouring recipes: [`23-pairwise-answer-evaluation`](../23-pairwise-answer-evaluation/),
  which this recipe's evaluation patterns build on, and
  [`38-abstention-and-review-queues`](../38-abstention-and-review-queues/), whose review-queue
  pattern this recipe's confidence gates follow.